1. Import dataset
- See variable description

In [26]:
import pandas as pd
import numpy as np

df = pd.read_excel("/content/Data_RS.xlsx")

2. Apply some filters
- Total capital has non-missing value

In [27]:
# Total capital = a2

df = df[df['a2'].notna()]  # .notna() is in the pandas package

3. Generate summary statistics
- Table 1, Panel B

In [28]:
vars = ["a", "a2", "oiadp_a2", "ppent_a2", "debt_mv", "d_a2", "mb"]

# Calculate statistics: Mean, SD, and Median
stats = df[vars].agg(['mean', 'std', 'median']).T  # .agg means 'aggregate' / .T means 'transpose'
stats.columns = ['Mean', 'SD', 'Median']

# Rename the index for better readability
stats.index = ["Book Assets", "Total Capital", "Profitability",
               "Tangibility", "Debt/Market Value", "Debt/Total Capital", "Market/Book"]

# Display the rounded results
display(stats.round(3))

,Mean,SD,Median
Book Assets,6184.601,17861.576,1304.838
Total Capital,4078.081,11408.309,926.359
Profitability,0.103,0.149,0.109
Tangibility,0.513,0.347,0.465
Debt/Market Value,0.263,0.194,0.238
Debt/Total Capital,0.502,0.343,0.478
Market/Book,1.849,1.364,1.420


4. Run regressions
- Table 3, Panel A
- Must include year-fixed effect

In [29]:
dep_vars = ["d_a2", "bankout_a2", "prog_a2", "bonds_a2", "pp_a2", "cv_a2", "mgeqoth_a2"]
dep_lables = ["Total Debt", "Bank", "Program", "Bonds", "PPs", "Convertibles", "All Other"]
key_vars = ["oiadp_a2", "ppent_a2", "mb", "lnsale", "(Intercept)"]
key_labels = ["Profitability", "Tangibility", "M/B", "ln(Sales)", "Constant"]

In [30]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Generate dummy variables (1997-2006)
year_dummies = pd.get_dummies(df['fyear'], prefix='year')
year_cols = [col for col in year_dummies.columns if any(str(y) in col for y in range(1997, 2007))]
year_fe_df = year_dummies[year_cols].astype(float)

# Merging with initial dataset
df_reg = pd.concat([df, year_fe_df], axis=1)

# 2. Remove missing value
reg_vars = ["oiadp_a2", "ppent_a2", "mb", "lnsale"]
df_reg = df_reg.dropna(subset=reg_vars + dep_vars)

# 3. Define regression formula
reg_formula = " + ".join(reg_vars) + " + " + " + ".join(year_cols)

# 4. Run regression
models_results = {}
for dv in dep_vars:
    formula = f"{dv} ~ {reg_formula}"
    # Cluster SE
    model = smf.ols(formula, data=df_reg).fit(cov_type='cluster', cov_kwds={'groups': df_reg['gvkey']})
    models_results[dv] = model

Replication of Rauh and Sufi (2010): Table 3, Panel A

In [31]:
def get_stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

table_rows = []
row_names = []

for kv, label in zip(key_vars, key_labels):
    est_row = []
    se_row = []
    for dv in dep_vars:
        res = models_results[dv]
        # Map '(Intercept)' to 'Intercept' for statsmodels naming
        var_name = 'Intercept' if kv == '(Intercept)' else kv

        coef = res.params[var_name]
        se = res.bse[var_name]
        p = res.pvalues[var_name]

        est_row.append(f"{coef:.3f}{get_stars(p)}")
        se_row.append(f"({se:.3f})")

    table_rows.append(est_row)
    table_rows.append(se_row)
    row_names.extend([label, ""])

# Add Adjusted R-Squared
r2_row = [f"{models_results[dv].rsquared_adj:.2f}" for dv in dep_vars]
table_rows.append(r2_row)
row_names.append("Adj. R-Squared")

# Create DataFrame
summary_table = pd.DataFrame(table_rows, index=row_names, columns=dep_lables)

# Display table
pd.set_option('display.width', 200)
print("Replication of Rauh and Sufi (2010): Table 3 Panel A")
display(summary_table)

Replication of Rauh and Sufi (2010): Table 3 Panel A


,Total Debt,Bank,Program,Bonds,PPs,Convertibles,All Other
Profitability,-0.549***,0.135*,0.007,-0.069,-0.108**,-0.422***,-0.092***
,(0.149),(0.072),(0.031),(0.093),(0.055),(0.110),(0.031)
Tangibility,0.158***,-0.019,0.044**,0.090***,0.025*,-0.045***,0.063**
,(0.038),(0.022),(0.017),(0.031),(0.014),(0.015),(0.025)
M/B,-0.031**,-0.025***,0.003,-0.024***,-0.001,0.020*,-0.004
,(0.012),(0.004),(0.003),(0.005),(0.004),(0.011),(0.003)
ln(Sales),-0.013,-0.026***,0.017***,-0.003,-0.005*,0.002,0.002
,(0.011),(0.005),(0.004),(0.007),(0.003),(0.004),(0.003)
Constant,0.550***,0.352***,-0.104***,0.150***,0.063***,0.058*,0.031
,(0.068),(0.035),(0.024),(0.049),(0.021),(0.030),(0.025)
